In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import time
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Ensure project root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.datasets import load_dataset
from src.ensemble import AdaptiveClusteringEnsemble
from src.models import KMeansMember

In [3]:
ds = load_dataset("wine")

X_raw = ds.X
X_scaled = ds.preprocessed()
y_true = ds.y

print(f"Dataset: {ds.name} | shape={X_raw.shape} | k={ds.k} | NaNs={np.isnan(X_raw).sum()}")
print(f"Raw feature ranges: min={X_raw.min(axis=0).round(2)} | max={X_raw.max(axis=0).round(2)}")

Dataset: Wine | shape=(178, 13) | k=3 | NaNs=0
Raw feature ranges: min=[1.103e+01 7.400e-01 1.360e+00 1.060e+01 7.000e+01 9.800e-01 3.400e-01
 1.300e-01 4.100e-01 1.280e+00 4.800e-01 1.270e+00 2.780e+02] | max=[1.483e+01 5.800e+00 3.230e+00 3.000e+01 1.620e+02 3.880e+00 5.080e+00
 6.600e-01 3.580e+00 1.300e+01 1.710e+00 4.000e+00 1.680e+03]


In [4]:
N_RUNS = 10
BASE_SEED = 0
SETTINGS = ("fixed", "variable")

rows = []

for variant, X in [("Unscaled", X_raw), ("Scaled (Z-score)", X_scaled)]:
    for setting in SETTINGS:
        variable_k = (setting == "variable")
        for run in range(N_RUNS):
            seed = BASE_SEED + run
            t0 = time.perf_counter()

            model = AdaptiveClusteringEnsemble(
                base_member=KMeansMember(),
                m_partitions=10,
                k_clusters=ds.k,
                variable_k=variable_k,
                sampling_ratio=0.7,
                alpha_1=0.8,
                alpha_2=0.7,
                delta_alpha=0.1,
                random_state=seed,
            )
            labels = model.fit_predict(X)

            rows.append({
                "variant": variant,
                "setting": setting,
                "run": run,
                "seed": seed,
                "ari": adjusted_rand_score(y_true, labels),
                "nmi": normalized_mutual_info_score(y_true, labels),
                "n_clusters_found": len(np.unique(labels)),
                "final_alpha_1": getattr(model, "final_alpha_1_", np.nan),
                "runtime_s": time.perf_counter() - t0,
            })

raw_df = pd.DataFrame(rows)
raw_df.head()

,variant,setting,run,seed,ari,nmi,n_clusters_found,final_alpha_1,runtime_s
0,Unscaled,fixed,0,0,0.374914,0.431543,3,0.8,0.247639
1,Unscaled,fixed,1,1,0.371114,0.428757,3,0.8,0.088974
2,Unscaled,fixed,2,2,0.336630,0.411055,3,0.8,0.084956
3,Unscaled,fixed,3,3,0.364545,0.424152,3,0.8,0.085862
4,Unscaled,fixed,4,4,0.371114,0.428757,3,0.8,0.084262


In [5]:
summary = (
    raw_df.groupby(["setting", "variant"])
    .agg(
        ari_mean=("ari", "mean"),
        ari_std=("ari", "std"),
        nmi_mean=("nmi", "mean"),
        nmi_std=("nmi", "std"),
    )
    .reset_index()
)

summary["ARI"] = summary["ari_mean"].map("{:.3f}".format) + " ± " + summary["ari_std"].map("{:.3f}".format)
summary["NMI"] = summary["nmi_mean"].map("{:.3f}".format) + " ± " + summary["nmi_std"].map("{:.3f}".format)

# Paper benchmarks for ACE on Wine (Tables 2, 3, 4, 5)
paper_rows = pd.DataFrame([
    {"setting": "fixed",    "variant": "Paper (ACE)", "ARI": "0.371 ± 0.008", "NMI": "0.429 ± 0.006"},
    {"setting": "variable", "variant": "Paper (ACE)", "ARI": "0.403 ± 0.014", "NMI": "0.421 ± 0.014"},
])

comparison = pd.concat(
    [paper_rows, summary[["setting", "variant", "ARI", "NMI"]]],
    ignore_index=True
).sort_values(["setting", "variant"]).reset_index(drop=True)

display(comparison)

,setting,variant,ARI,NMI
0,fixed,Paper (ACE),0.371 ± 0.008,0.429 ± 0.006
1,fixed,Scaled (Z-score),0.890 ± 0.028,0.872 ± 0.026
2,fixed,Unscaled,0.367 ± 0.011,0.427 ± 0.006
3,variable,Paper (ACE),0.403 ± 0.014,0.421 ± 0.014
4,variable,Scaled (Z-score),0.815 ± 0.104,0.814 ± 0.079
5,variable,Unscaled,0.282 ± 0.119,0.306 ± 0.129


In [6]:
out_dir = os.path.join(PROJECT_ROOT, "results", "wine_unscaled")
os.makedirs(out_dir, exist_ok=True)

raw_df.to_csv(os.path.join(out_dir, "raw_results.csv"), index=False)
comparison.to_csv(os.path.join(out_dir, "comparison_table.csv"), index=False)
print(f"Saved to {out_dir}")

Saved to /media/marta/Dane/Sem9/FuzzyAdaptiveClusteringEnsemble/results/wine_unscaled
